### NOMEANDO VARIÁVEIS

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Reaproveitando o mesmo padrão de nomeação de schemas usado na Bronze e na Silver
catalog = "workspace"

# Usando os mesmos nomes do Landing_to_Bronze para as três databases (schemas) 
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

# Novamente guardando o caminho completo de cada schema (catalog + nome do schema).
bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Print pra validar
print(f"gold_schema: {gold_schema}")

gold_schema: workspace.gold


### CRIA DATABASE SE NAO EXISTIR

In [0]:
# Cria o database (schema) gold se ele ainda não existir
spark.sql(f"CREATE DATABASE IF NOT EXISTS {gold_schema}")

print(f"gold_schema: {gold_schema}")
print(f"Database {gold_schema} pronto.")

gold_schema: workspace.gold
Database workspace.gold pronto.


### Pq não vai ter `dim_date`?
- Foi mostrado na aula de guilherme sobre camada gold q é importante logo no início definir uma dimensão data para guardar as datas no formato yyyymmdd
- Ao meu ver, isso faz sim sentido no exemmplo usado na aula que verifica as ordens de vendas e pedidos, já q pra um dia existem muuuuuitos pedidos e vendas
- O mesmo nao acontece nessa atividade dos filmes. As datas nas tabelas de filme estão relacionadas a data de lançamento e ano de lançamento. 
- Como no próprio .pdf diz "Grão: Um registro único por filme.", o registro é por filme, nao por dia. Além disso o documento não pede dim_date, só pede dim_movies, dim_genres, dim_people, dim_companies, dim_reviews, fact_movies_performance e as 3 bridges

### Outra questão é em relação as views
- Na aula foi mostrado Views e Materialized Views, mas no documnento diz:
- "Utilize exibições de DataFrames em tela (display()) para responder às perguntas de negócio solicitadas. Nenhuma biblioteca de visualização de gráficos é necessária."
- Por isso, entendi que nao é preciso criar as views, os resultados são todos exibidos no notebook via display()

Deixo esse comentário registrado para deixar claro que a ausência de dim_date e Views é uma decisão tomada levando em consideração o escopo do projeto, não uma lacuna

### Organização do Star Schema
![image_1789929394738.png](./image_1789929394738.png "image_1789929394738.png")

### CRIAÇÃO DO `gold.dim_movies`

In [0]:
# CARREGANDO A TABELA silver.tb_info_filmes PRA CRIAR A dim_movies

# CRIANDO PRIMEIRO A dim_movies PQ TEM A PK QUE VAI SER USADA COMO FK NAS OUTRAS dim

df_silver_info_filmes = spark.table(f"{silver_schema}.tb_info_filmes")

# Criando a surrogate key sk_movie_id, começa em 1 e vai incrementando de 1 em 1 (n é o msm valor q id_filme) 
# Selecionando as colunas definidas no documento da atividade
df_dim_movies = (
    df_silver_info_filmes

    .withColumn("sk_movie_id", F.row_number().over(Window.orderBy("id_filme")))
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse",
    )
)

# Gravando com overwrite normal
df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

#print pra validar e display
print(f"dim_movies: {df_dim_movies.count()} filmes")
display(df_dim_movies.limit(10))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_movies: 97468 filmes


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
1,14564,Rings,2017-02-01,2017,102,en,Lançado,"Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
2,32471,Mixtape,2021-12-03,2021,94,en,Lançado,null
3,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,All hell breaks loose when a giant grizzly
4,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
5,38700,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
6,42018,The Horse Thief,2019-03-19,2019,88,zh,Lançado,"Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
7,42330,Monkey Magic,2018-09-22,2018,66,zh,Lançado,"Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
8,43074,Ghostbusters,2016-07-14,2016,117,en,Lançado,"Following a ghost invasion of Manhattan, paranormal enthusiasts Erin Gilbert and Abby Yates, nuclear engineer Jillian Holtzmann, and subway worker Patty Tolan band together to stop the otherworldly threat."
9,45033,20 Seconds of Joy,2018-01-01,2018,60,de,Lançado,"Traces the story of an extreme athlete, past and present; but also explores the psychology behind life, death, risk and the confrontation of fear."
10,46983,The Song of Styrene,2022-05-23,2022,13,fr,Lançado,Le chant du Styrène is a 1958 French documentary film directed by Alain Resnais. The film was an order by French industrial group Pechiney to highlight the merits of plastics.


### CRIAÇÃO DO `gold.dim_genres`

In [0]:
# CARREGANDO A TABELA silver.tb_generos PRA CRIAR A dim_genres

# SEGUINDO COM A genres ANTES DA fact_movies_performance PQ QUERO CRIAR PRIMEIRO AQUELAS QUE DEPENDEM SOMENTE DA SILVER, POSTERIORMENTE CRIO AS QUE PRECISAM DE UMA dim PRÉVIA

df_silver_generos = spark.table(f"{silver_schema}.tb_generos")

# Pegando só os nomes de gênero ÚNICOS 
# dim_genres precisa ter 1 linha por genero (é como se fosse uma lista de todos os generos possíveis)
df_generos_unicos = df_silver_generos.select("genero").distinct()

# Guardando dim_genres
df_dim_genres = (
    df_generos_unicos
    .withColumn("sk_genre_id", F.row_number().over(Window.orderBy("genero")))
    .select(
        "sk_genre_id",
        F.col("genero").alias("nome_genero"),
    )
)

# Gravando com overwrite normal
df_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_genres")

# Print pra validar e display
print(f"dim_genres: {df_dim_genres.count()} gêneros únicos")
display(df_dim_genres.orderBy("sk_genre_id"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_genres: 19 gêneros únicos


sk_genre_id,nome_genero
1,Action
2,Adventure
3,Animation
4,Comedy
5,Crime
6,Documentary
7,Drama
8,Family
9,Fantasy
10,History


### `gold.dim_people` e `gold.dim_companies`

- Aqui é preciso dividir silver.tb_pessoas_empresas em 2 dimensões
- Os tipo Ator, Diretor, Roteirista, Produtora vão ser divididos em 2 dimensões
- `gold.dim_people`: pessoa e o tipo da pessoa
- `gold.dim_companies`: 

### CRIAÇÃO DA `gold.dim.people`

In [0]:
# CARREGANDO A TABELA silver_pessoas_empresas PRA CRIAR A dim_people

df_silver_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

# Filtrando apenas os 3 tipos de pessoa, excluindo produtora
# .distinct() por (nome, tipo): a mesma pessoa pode ter mais de um papel (ex: alguém que é Ator E Diretor no mesmo filme, ou em filmes diferentes)
df_pessoas_unicas = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") != "Produtora")
    .select(
        F.col("nome_pessoa_empresa").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa"),
    )
    .distinct()
)

# Guardando dim_people
df_dim_people = (
    df_pessoas_unicas
    .withColumn("sk_person_id", F.row_number().over(Window.orderBy("nome_pessoa", "tipo_pessoa")))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

# Gravando em overwrite normal
df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

# Print pra validar e display
print(f"dim_people: {df_dim_people.count()} pessoas")
display(df_dim_people.limit(200))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_people: 418972 pessoas


sk_person_id,nome_pessoa,tipo_pessoa
1,'Ana Ika,Ator
2,'E-gotti' Eric Johnson,Ator
3,'Jeeva' Ravi,Ator
4,'Meesai' Mohan,Ator
5,'Meesai' Rajendran,Ator
6,'Om' Rakesh Chaturvedi,Ator
7,'Poo' Ram,Ator
8,'Sunday Jeff' Silverman,Ator
9,'Weird Al' Yankovic,Roteirista
10,'Wáats'asdíyei Joe Yates,Diretor


### CRIAÇÃO DA `gold.dim.companies`

Identifiquei alguns nomes de companias errados que começam#, &, ', (), pois inspeção manual mostrou que vários desses casos são estilização intencional do nome (ex: jogos de palavra, marcas), e uma correção automática arriscaria descaracterizar nomes reais — mesma lição aprendida na limpeza de título.

In [0]:
# APROVEITANDO A TABELA silver_pessoas_empresas CARREGADA NA CEDULA ANTERIOR PRA CRIAR A dim_companies

# Filtrando apenas o tipo prodtora (o inverso do filtro usado na dim_people)
df_produtoras_unicas = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_pessoa_empresa").alias("nome_produtora"))
    .distinct()
)

# Guardando dim_companies
df_dim_companies = (
    df_produtoras_unicas
    .withColumn("sk_company_id", F.row_number().over(Window.orderBy("nome_produtora")))
    .select("sk_company_id", "nome_produtora")
)

# Gravando em overwrite normal
df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

# Print pra validar e display
print(f"dim_companies: {df_dim_companies.count()} produtoras")
display(df_dim_companies.limit(200))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_companies: 44910 produtoras


sk_company_id,nome_produtora
1,#1nfluence Production
2,#beardforce Films
3,#sinning Works
4,& Extermination In An American City
5,& Space Productions
6,'S Wonderful Pictures
7,((o))eco
8,(mark Paul Wake
9,(not) Heroine Movies
10,(notice Me) Kid Vicious


### CRIAÇÃO DA `gold.dim_reviews`

In [0]:
# CARREGANDO tb_avaliacoes_usuarios E dim_movies (pro lookup do sk_movie_id)
df_silver_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
df_dim_movies_lookup = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

# 1 linha por avaliação pra 1 linha por filme
df_avaliacoes_agregadas = (
    df_silver_avaliacoes
    .groupBy("id_filme")
    .agg(
        F.count("*").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")  # Documento pede arredondado em 2 casas
    )
)

# JOIN com dim_movies: partimos de TODOS os filmes (LEFT), não só dos que têm avaliação, pra nenhum filme "sumir" da dimensão por falta de review
df_dim_reviews = (
    df_dim_movies_lookup
    .join(df_avaliacoes_agregadas, on="id_filme", how="left")
    .withColumn("sk_review_id", F.row_number().over(Window.orderBy("sk_movie_id")))
    # Tratamento de nulos: filme sem review tem 0 avaliações de verdade, então o nulo vira 0.
    # JÁ nota_media_usuarios continua nula de propósito: não existe "nota 0" pra quem não foi avaliado,
    # preencher com 0 seria mentir que o filme foi avaliado e tirou nota mínima.
    .withColumn("qtd_avaliacoes_usuarios", F.coalesce(F.col("qtd_avaliacoes_usuarios"), F.lit(0)))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

# Gravando em overwrite normal
df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

# Print pra validar e display
print(f"dim_reviews: {df_dim_reviews.count()} filmes (todos, com ou sem avaliações)")
display(df_dim_reviews.limit(100))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


dim_reviews: 97468 filmes (todos, com ou sem avaliações)


sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
1,1,0,null
2,2,1,8.3
3,3,2,5.15
4,4,1,2.5
5,5,1,0.1
6,6,0,null
7,7,0,null
8,8,0,null
9,9,0,null
10,10,1,0.2


### CRIAÇÃO DA `gold.dim_fact_movies_performance`

In [0]:
# CARREGANDO AS TABELAS SILVER DE ORIGEM DA FATO
# (sem isso, o notebook não roda numa sessão nova -- essas variáveis só existiam em memória)
df_fin = spark.table(f"{silver_schema}.tb_financeiro_filmes")
df_eng = spark.table(f"{silver_schema}.tb_metricas_engajamento")

# LOOKUP: sk_movie_id de cada filme 
df_fact_lookup = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

# JOIN - Usa das métricas financeiras e de engajamento na base de filmes
# LEFT JOIN - nenhum filme é perdido, mesmo sem dado financeiro/engajamento
# Sem coalesce: nulo aqui = "não sabemos", não "zero" -> já tratado corretamente na Silver
df_fact_movies_performance = (
    df_fact_lookup
    .join(df_fin, on="id_filme", how="left")
    .join(df_eng, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        "orcamento_usd", "receita_usd", "lucro_usd",
        "orcamento_brl", "receita_brl", "lucro_brl",
        "margem_lucro_percentual",
        "popularidade", "nota_media_tmdb", "qtd_votos_tmdb",
        "nota_media_imdb", "qtd_votos_imdb"
    )
)

# Gravando em overwrite normal
df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

# Print pra validar e display
print(f"fact_movies_performance: {df_fact_movies_performance.count()} filmes")
display(df_fact_movies_performance.limit(100))

fact_movies_performance: 97468 filmes


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,margem_lucro_percentual,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1,2.5E7,8.308089E7,5.808089E7,1.289225E8,4.2843984164E8,2.9951734164E8,232.32,24.584,4.966,2375,null,46286
2,null,null,null,null,null,null,null,8.929,7.064,118,6.6,4617
3,7500000.0,null,null,3.867675E7,null,null,null,null,3.161,28,null,null
4,null,null,null,null,null,null,null,2.98,7.1,null,7.8,212
5,9.0E7,4.26505244E8,3.36505244E8,4.64121E8,2.19944489278E9,1.73532389278E9,373.89,46.619,7.139,7570,6.5,199420
6,null,null,null,null,null,null,null,3.403,6.63,27,6.8,1750
7,null,null,null,null,null,null,null,1.561,10.0,1,6.4,34
8,1.44E8,2.29147509E8,8.5147509E7,7.425936E8,1.18169078916E9,4.3909718916E8,59.13,40.052,5.371,5976,null,260417
9,337200.0,null,null,1738906.68,null,null,null,0.6,8.0,2,null,154
10,null,null,null,null,null,null,null,0.693,8.5,2,7.0,1209


### CRIANDO AS BRIDGES `bridge_movie_genre`, `bridge_movie_person` E `bridge_movie_company`

### `bridge_movie_genre`

In [0]:
# CRIANDO BRIDGE movie_genre

# LOOKUPS: SKs das duas dimensões que essa bridge conecta
df_movies_lookup = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")
df_genres_lookup = spark.table(f"{gold_schema}.dim_genres")  # sk_genre_id, nome_genero

# Troca id_filme -> sk_movie_id e genero -> sk_genre_id, ficando só com as duas FKs
df_bridge_movie_genre = (
    df_silver_generos
    .join(df_movies_lookup, on="id_filme", how="inner")
    .join(df_genres_lookup, df_silver_generos["genero"] == df_genres_lookup["nome_genero"], how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()  # remove duplicata exata de filme-gênero, se houver
)

# Gravando em overwrite normal
df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

# Print pra validar e display
print(f"bridge_movie_genre: {df_bridge_movie_genre.count()} combinações filme-gênero")
display(df_bridge_movie_genre.limit(20))

bridge_movie_genre: 139774 combinações filme-gênero


sk_movie_id,sk_genre_id
15,14
18,9
19,7
20,2
39,11
40,15
56,11
85,7
86,5
99,7


### `bridge_movie_person`

In [0]:
# CRIANDO BRIDGE movie_person

df_people_lookup = spark.table(f"{gold_schema}.dim_people")  # sk_person_id, nome_pessoa, tipo_pessoa

df_bridge_movie_person = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") != "Produtora")
    .join(df_movies_lookup, on="id_filme", how="inner")
    .join(
        df_people_lookup,
        (df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_people_lookup["nome_pessoa"]) &
        (df_silver_pessoas_empresas["tipo_entidade"] == df_people_lookup["tipo_pessoa"]),
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

# Gravando em overwrite normal
df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

# Print pra validar e display
print(f"bridge_movie_person: {df_bridge_movie_person.count()} combinações filme-pessoa")
display(df_bridge_movie_person.limit(20))

bridge_movie_person: 753819 combinações filme-pessoa


sk_movie_id,sk_person_id
4989,160617
11393,104529
12219,271561
12450,310531
17379,188339
18892,213861
25624,59786
31033,264934
32774,82546
34968,222830


### `bridge_movie_company`

In [0]:
# CRIANDO BRIDGE movie_company

df_companies_lookup = spark.table(f"{gold_schema}.dim_companies")  # sk_company_id, nome_produtora

df_bridge_movie_company = (
    df_silver_pessoas_empresas
    .filter(F.col("tipo_entidade") == "Produtora")
    .join(df_movies_lookup, on="id_filme", how="inner")
    .join(
        df_companies_lookup,
        df_silver_pessoas_empresas["nome_pessoa_empresa"] == df_companies_lookup["nome_produtora"],
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

# Gravando em overwrite normal
df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

# Print pra validar e display
print(f"bridge_movie_company: {df_bridge_movie_company.count()} combinações filme-produtora")
display(df_bridge_movie_company.limit(20))

bridge_movie_company: 117602 combinações filme-produtora


sk_movie_id,sk_company_id
91,40838
149,22023
189,6672
273,15614
277,24900
283,23086
334,38987
365,18190
517,7871
654,29614


### CRIANDO TABELA `gold_genai_movies_context`


In [0]:
# Definindo base de filmes de dados como dim_movies + métricas financeiras da fact
df_movies_base = (
    spark.table(f"{gold_schema}.dim_movies")
    .join(
        spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "receita_usd", "orcamento_usd"),
        on="sk_movie_id", how="left"
    )
)

# Definindo base de autores como bridge_movie_person + dim_people (tipo_pessoa == 'Ator')
df_atores = (
    spark.table(f"{gold_schema}.bridge_movie_person")
    .join(df_people_lookup.filter(F.col("tipo_pessoa") == "Ator"), on="sk_person_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("atores_principais"))
)

# Definindo base de diretores como bridge_movie_person + dim_people (tipo_pessoa == 'Diretor')
df_diretores = (
    spark.table(f"{gold_schema}.bridge_movie_person")
    .join(df_people_lookup.filter(F.col("tipo_pessoa") == "Diretor"), on="sk_person_id", how="inner")
    .groupBy("sk_movie_id")
    .agg(F.concat_ws(", ", F.collect_list("nome_pessoa")).alias("diretor"))
)

# Juntando todas as bases em um so contexto
df_contexto = (
    df_movies_base
    .join(df_atores, on="sk_movie_id", how="left")
    .join(df_diretores, on="sk_movie_id", how="left")
)

# Monta o texto final, com fallback todo campo que pode faltar
df_gold_genai_movies_context = df_contexto.select(
    F.col("id_filme").alias("movie_id"),
    F.col("titulo").alias("title"),
    F.concat(
        F.lit("O filme "), F.coalesce(F.col("titulo"), F.lit("Título desconhecido")),
        F.lit(", lançado no ano de "), F.coalesce(F.col("ano_lancamento").cast("string"), F.lit("ano desconhecido")),
        F.lit(", faturou "), F.coalesce(F.format_number(F.col("receita_usd"), 2), F.lit("valor de receita indisponível")),
        F.lit(" e teve um custo de "), F.coalesce(F.format_number(F.col("orcamento_usd"), 2), F.lit("valor de orçamento indisponível")),
        F.lit(". Estrelado por "), F.coalesce(F.col("atores_principais"), F.lit("elenco não divulgado")),
        F.lit(" e dirigido por "), F.coalesce(F.col("diretor"), F.lit("diretor não divulgado")),
        F.lit(", o filme possui a seguinte sinopse: "), F.coalesce(F.col("sinopse"), F.lit("sinopse não disponível."))
    ).alias("llm_context_document")
)

# Gravando em overwrite normal
df_gold_genai_movies_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.gold_genai_movies_context")

# Print pra validar e display
print(f"gold_genai_movies_context: {df_gold_genai_movies_context.count()} filmes")
display(df_gold_genai_movies_context.limit(15))

gold_genai_movies_context: 97468 filmes


movie_id,title,llm_context_document
14564,Rings,"O filme Rings, lançado no ano de 2017, faturou 83,080,890.00 e teve um custo de 25,000,000.00. Estrelado por Chuck David Willis, Bonnie Morgan, Matilda Lutz, Aimee Teegarden, Vincent D'onofrio, Laura Slade Wiggins, Zach Roerig, Johnny Galecki, Alex Roe, Patrick Walker e dirigido por F. Javier Gutiérrez, o filme possui a seguinte sinopse: Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
32471,Mixtape,"O filme Mixtape, lançado no ano de 2021, faturou valor de receita indisponível e teve um custo de valor de orçamento indisponível. Estrelado por Kiefer O'reilly, Julie Bowen, Audrey Hsieh, Jackson Rathbone, Gemma Brooke Allen, Lucas Yao, Olga Petsa, Nick Thune, Anthony Timpano e dirigido por Valerie Weiss, o filme possui a seguinte sinopse: sinopse não disponível."
38258,Grizzly II: Revenge,"O filme Grizzly II: Revenge, lançado no ano de 2020, faturou valor de receita indisponível e teve um custo de 7,500,000.00. Estrelado por elenco não divulgado e dirigido por English, o filme possui a seguinte sinopse: All hell breaks loose when a giant grizzly"
38492,Billy Joel - Live at Yankee Stadium,"O filme Billy Joel - Live at Yankee Stadium, lançado no ano de 2022, faturou valor de receita indisponível e teve um custo de valor de orçamento indisponível. Estrelado por Tommy Byrnes, Jeffrey Jacobs, Mark Rivera, Billy Joel, Liberty Devitto, David Brown, Schuyler Deale e dirigido por Jon Small, o filme possui a seguinte sinopse: Billy Joel plays his greatest hits in the Big Apple."
38700,Bad Boys for Life,"O filme Bad Boys for Life, lançado no ano de 2020, faturou 426,505,244.00 e teve um custo de 90,000,000.00. Estrelado por Kate Del Castillo, Paola Nuñez, Nicky Jam, Vanessa Hudgens, Jacob Scipio, Martin Lawrence, Joe Pantoliano, Charles Melton, Alexander Ludwig, Will Smith e dirigido por Adil El Arbi, Bilall Fallah, o filme possui a seguinte sinopse: Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."
42018,The Horse Thief,"O filme The Horse Thief, lançado no ano de 2019, faturou valor de receita indisponível e teve um custo de valor de orçamento indisponível. Estrelado por Drashi, Jamco Jayang, Daiba, Rigzin Tseshang, Jiji Dan, Gaoba e dirigido por Zhuangzhuang Tian, Peicheng Pan, o filme possui a seguinte sinopse: Devout Buddhists, Norbu and Dolma live with their young son Tashi in a clan in Tibet. Norbu is a highwayman. After Norbu is charged with stealing from the temple, he and his family are banished. Impoverished and marginalized, they can do little when their beloved son becomes ill. Tashi dies of a fever. After a second son is born, Norbu focuses his every action on keeping this child alive, seeking re-admission to the clan for his wife and child, then risking all to save them from isolation and starvation in winter."
42330,Monkey Magic,"O filme Monkey Magic, lançado no ano de 2018, faturou valor de receita indisponível e teve um custo de valor de orçamento indisponível. Estrelado por Liu Beichen, Jihai Ma, Ye Sun, Dian Tao e dirigido por diretor não divulgado, o filme possui a seguinte sinopse: Dearth Voyd, the ruler of the dark side of the universe, wants to turn the human world into a place of violence and evil!! However, a huge meteor falls upon Flower-Fruit mountain giving birth to Kongo, a monkey made of stone! Kongo, now sets out on his quest to fight evil and restore order to a desperate land. Episodes 1-3"
43074,Ghostbusters,"O filme Ghostbusters, lança

### Perguntas de Negócio:
- 1. Qual é a receita total (em R$) somada de todos os filmes da base?
- 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.
- 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.
- 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).
- 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?
- 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?
-
- *Para o recorte dos últimos 2 e 5 anos, considere como data limite superior a data de lançamento realizada mais recente na base (ignorando registros com datas futuras ou não lançadas). 


### 1. Qual é a receita total (em R$) somada de todos os filmes da base?

In [0]:
# PERGUNTA 1: Receita total (em R$) de todos os filmes da base
df_receita_total = (
    spark.table(f"{gold_schema}.fact_movies_performance")
    .agg(F.sum("receita_brl").alias("receita_total_brl"))
)

display(df_receita_total)

receita_total_brl
8.347305110685377E11


### 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.

In [0]:
# PERGUNTA 2: Top 5 filmes com maior popularidade
df_top_popularidade = (
    spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "titulo")
    .join(
        spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "popularidade"),
        on="sk_movie_id", how="inner"
    )
    .select("titulo", "popularidade")
    .orderBy(F.col("popularidade").desc())
    .limit(5)
)

display(df_top_popularidade)

titulo,popularidade
blue beetle,2994.357
Gran Turismo,2680.593
The Nun II,1692.778
Meg 2: The Trench,1567.273
retribution,1547.22


### 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.

In [0]:
# PERGUNTA 3: Quantidade de filmes por gênero, do maior pro menor
df_filmes_por_genero = (
    spark.table(f"{gold_schema}.bridge_movie_genre")
    .join(spark.table(f"{gold_schema}.dim_genres"), on="sk_genre_id", how="inner")
    .groupBy("nome_genero")
    .agg(F.count("*").alias("qtd_filmes"))
    .orderBy(F.col("qtd_filmes").desc())
)

display(df_filmes_por_genero)

nome_genero,qtd_filmes
Drama,32235
Documentary,18959
Comedy,18523
Thriller,10226
Horror,9635
Romance,7634
Action,5975
Crime,4720
Animation,4406
TV Movie,4075


### 4. Para os 10 filmes de maior receita, mostre título, receita `em US$ e R$` e a posição de cada um no ranking `RANK()`.

In [0]:
from pyspark.sql import Window

# PERGUNTA 4: Top 10 filmes de maior receita (US$ e R$) com posição no ranking
janela_receita = Window.orderBy(F.col("receita_usd").desc())

df_top_receita = (
    spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "titulo")
    .join(
        spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "receita_usd", "receita_brl"),
        on="sk_movie_id", how="inner"
    )
    .withColumn("posicao_ranking", F.rank().over(janela_receita))
    .select("posicao_ranking", "titulo", "receita_usd", "receita_brl")
    .orderBy("posicao_ranking")
    .limit(10)
)

display(df_top_receita)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


posicao_ranking,titulo,receita_usd,receita_brl
1,Avengers: Endgame,2.8E9,1.443932E10
2,Avatar: The Way of Water,2.320250281E9,1.196529867409E10
3,AVENGERS: INFINITY WAR,2.052415039E9,1.058409911462E10
4,spider-man: no way home,1.921847111E9,9.91077336672E9
5,The Lion King,1.663075401E9,8.57631353542E9
6,Top Gun: Maverick,1.488732821E9,7.67724628461E9
7,Barbie,1.428545028E9,7.36686385489E9
8,The Super Mario Bros. Movie,1.355725263E9,6.99133960876E9
9,Black Panther,1.349926083E9,6.96143381742E9
10,Star Wars: The Last Jedi,1.33269883E9,6.87259459643E9


### Observação sobre o recorte temporal (Perguntas 5 e 6)

A `data_maxima` calculada foi **2026-02-19**. Investigando a distribuição de filmes por ano de lançamento, a base tem cerca de 10 mil filmes por ano até 2023, cai para 1.800 em 2024 e despenca para 5 (2025), 1 (2026), 2 (2027) e 1 (2029). Indicando que a base foi construída por volta de meados de 2024, e que os registros posteriores são residuais.

**Consequência:** a janela de 2 anos definida por essa data máxima (2024-02-19 a 2026-02-19) cai justamente na região mais esparsa dos dados, contendo apenas 874 filmes dos 97.468 da base. Isso explica o número baixo de participações do ator no topo do ranking (4).


Mantive a regra exatamente como o documento pede: 
Data de lançamento realizada mais recente na base, ignorando registros com datas futuras ou não lançadas — e registro aqui que o resultado é sensível a esse artefato da base. Vale notar também que, como o filtro de "data futura" depende da data de execução (`current_date()`), o recorte pode variar conforme o momento em que o notebook é executado.

### 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?

In [0]:
# PERGUNTA 5: Ator com mais participações nos filmes lançados nos últimos 2 anos
# "Últimos 2 anos" = a partir da data de lançamento mais recente da PRÓPRIA base
# (não da data de hoje), ignorando filmes com data futura ou não lançados (status_filme)

df_movies_validos = spark.table(f"{gold_schema}.dim_movies").filter(
    (F.col("data_lancamento").isNotNull()) &
    (F.col("data_lancamento") <= F.current_date()) &
    (F.col("status_filme") == "Lançado")
)

data_maxima = df_movies_validos.agg(F.max("data_lancamento")).collect()[0][0]
print(f"Data de lançamento mais recente válida na base: {data_maxima}")

data_limite_inferior = F.add_months(F.lit(data_maxima), -24)  # 2 anos = 24 meses antes da data máxima

df_filmes_ultimos_2_anos = df_movies_validos.filter(F.col("data_lancamento") >= data_limite_inferior)

# Conta participações de ATORES nesses filmes, via bridge_movie_person + dim_people
df_ator_top_2_anos = (
    df_filmes_ultimos_2_anos.select("sk_movie_id")
    .join(spark.table(f"{gold_schema}.bridge_movie_person"), on="sk_movie_id", how="inner")
    .join(df_people_lookup.filter(F.col("tipo_pessoa") == "Ator"), on="sk_person_id", how="inner")
    .groupBy("nome_pessoa")
    .agg(F.count("*").alias("qtd_participacoes"))
    .orderBy(F.col("qtd_participacoes").desc())
    .limit(1)
)

display(df_ator_top_2_anos)

Data de lançamento mais recente válida na base: 2026-02-19


nome_pessoa,qtd_participacoes
Suhas,4


### 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

In [0]:
# PERGUNTA 6: Produtora com maior lucro nos últimos 5 anos (mesma data-limite: lançamento mais recente válido da base)
data_limite_inferior_5anos = F.add_months(F.lit(data_maxima), -60)  # 5 anos = 60 meses antes da data máxima

df_filmes_ultimos_5_anos = df_movies_validos.filter(F.col("data_lancamento") >= data_limite_inferior_5anos)

df_companies_lookup = spark.table(f"{gold_schema}.dim_companies")

df_produtora_top_lucro_5anos = (
    df_filmes_ultimos_5_anos.select("sk_movie_id")
    .join(spark.table(f"{gold_schema}.bridge_movie_company"), on="sk_movie_id", how="inner")
    .join(df_companies_lookup, on="sk_company_id", how="inner")
    .join(spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "lucro_usd"), on="sk_movie_id", how="inner")
    .groupBy("nome_produtora")
    .agg(F.sum("lucro_usd").alias("lucro_total_usd"))
    .orderBy(F.col("lucro_total_usd").desc())
    .limit(1)
)

display(df_produtora_top_lucro_5anos)

nome_produtora,lucro_total_usd
Universal Pictures,5.772329679E9


### testes

In [0]:
display(
    spark.table(f"{gold_schema}.dim_people")
    .filter((F.col("nome_pessoa") == "Suhas") & (F.col("tipo_pessoa") == "Ator"))
    .join(spark.table(f"{gold_schema}.bridge_movie_person"), on="sk_person_id")
    .join(spark.table(f"{gold_schema}.dim_movies"), on="sk_movie_id")
    .filter(
        (F.col("data_lancamento") <= F.lit(data_maxima)) &
        (F.col("data_lancamento") >= F.add_months(F.lit(data_maxima), -24))
    )
    .select("titulo", "data_lancamento")
    .orderBy("data_lancamento")
)

titulo,data_lancamento
Sriranga Neethulu,2024-04-11
Prasanna Vadanam,2024-05-03
Gorre Puranam,2024-09-20
Janaka Aithe Ganaka,2024-10-12


In [0]:
for tabela in ["dim_movies", "dim_reviews", "fact_movies_performance", "gold_genai_movies_context"]:
    print(f"{tabela}: {spark.table(f'{gold_schema}.{tabela}').count()}")

dim_movies: 97468
dim_reviews: 97468
fact_movies_performance: 97468
gold_genai_movies_context: 97468


In [0]:
dm = spark.table(f"{gold_schema}.dim_movies")

# Distribuição de filmes por ano nos anos recentes
display(
    dm.filter(F.col("ano_lancamento") >= 2015)
      .groupBy("ano_lancamento")
      .agg(F.count("*").alias("qtd_filmes"))
      .orderBy("ano_lancamento")
)

# Quanto cada filtro da Pergunta 5 exclui
print(f"total dim_movies: {dm.count()}")
print(f"com data_lancamento nula: {dm.filter(F.col('data_lancamento').isNull()).count()}")
print(f"com data futura (> hoje): {dm.filter(F.col('data_lancamento') > F.current_date()).count()}")
print(f"com status != 'Lançado': {dm.filter(F.col('status_filme') != 'Lançado').count()}")

ano_lancamento,qtd_filmes
2016,12356
2017,13318
2018,13292
2019,13633
2020,11163
2021,11216
2022,11570
2023,9047
2024,1800
2025,5


total dim_movies: 97468
com data_lancamento nula: 64
com data futura (> hoje): 3
com status != 'Lançado': 1415


In [0]:
# Validação completa e comparativa da Tabela Fato com os títulos da Dimensão
display(
    spark.sql(f"""
        SELECT 
            m.id_filme,
            m.titulo,
            f.sk_movie_id,
            f.orcamento_usd,
            f.receita_usd,
            f.lucro_usd,
            f.orcamento_brl,
            f.receita_brl,
            f.lucro_brl
        FROM {gold_schema}.fact_movies_performance f
        JOIN {gold_schema}.dim_movies m ON f.sk_movie_id = m.sk_movie_id
        ORDER BY CAST(m.id_filme AS INT) ASC
        LIMIT 20
    """)
)

id_filme,titulo,sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl
14564,Rings,1,2.5E7,8.308089E7,5.808089E7,1.289225E8,4.2843984164E8,2.9951734164E8
32471,Mixtape,2,null,null,null,null,null,null
38258,Grizzly II: Revenge,3,7500000.0,null,null,3.867675E7,null,null
38492,Billy Joel - Live at Yankee Stadium,4,null,null,null,null,null,null
38700,Bad Boys for Life,5,9.0E7,4.26505244E8,3.36505244E8,4.64121E8,2.19944489278E9,1.73532389278E9
42018,The Horse Thief,6,null,null,null,null,null,null
42330,Monkey Magic,7,null,null,null,null,null,null
43074,Ghostbusters,8,1.44E8,2.29147509E8,8.5147509E7,7.425936E8,1.18169078916E9,4.3909718916E8
45033,20 Seconds of Joy,9,337200.0,null,null,1738906.68,null,null
46983,The Song of Styrene,10,null,null,null,null,null,null
